---
title: "Final Project: Architecture Integration Capstone"
format: html
---


# Day 5 — Architecture Integration Capstone

All five modules integrated: Kafka ingestion → Bronze/Silver/Gold Lakehouse → Quality Gate → RAG serving → Orchestration DAG with lineage.

In [1]:
!pip install pandas chromadb sentence-transformers loguru

In [2]:
import os
import json
import uuid
import random
from datetime import datetime, timedelta
from collections import defaultdict

import pandas as pd
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from sentence_transformers import SentenceTransformer
from loguru import logger

logger.add("capstone_run.log", rotation="10 MB")

BRONZE_PATH = "./lakehouse/bronze"
SILVER_PATH = "./lakehouse/silver"
GOLD_PATH   = "./lakehouse/gold"

In [3]:
# =====================================================================
# MODULE 1 — INGESTION (Kafka simulation)

In [4]:
# =====================================================================

class MockKafka:
    """
    Simulates Kafka's topic/partition/offset model.

    In real Kafka:
      - Topics are split into partitions (parallel ordered logs).
      - Producers assign events to partitions (round-robin or key-hash).
      - Consumer groups share partitions — each partition to one member.
      - Offsets are committed so consumers can resume after failure.
    """

    def __init__(self):
        self.topics  = defaultdict(list)
        self.offsets = defaultdict(int)

    def produce(self, topic: str, event: dict):
        self.topics[topic].append({
            "offset":     self.offsets[topic],
            "event_time": datetime.utcnow().isoformat(),
            "payload":    event,
        })
        self.offsets[topic] += 1

    def consume_all(self, topic: str) -> list[dict]:
        return self.topics.pop(topic, [])


def module1_ingest(kafka: MockKafka) -> int:
    """
    Produces synthetic e-commerce order events to the orders_raw topic.
    Replace this function with a real Kafka producer for your project.
    """
    logger.info("[MODULE 1] Ingestion — producing events to Kafka...")

    products  = ["Laptop", "Mouse", "Keyboard", "Monitor", "Headset", "Webcam"]
    countries = ["SA", "AE", "EG", "KW", "BH", "QA"]

    for i in range(25):
        kafka.produce("orders_raw", {
            "order_id":    f"ORD-{1000 + i:04d}",
            "customer_id": f"CUST-{random.randint(1, 10):03d}",
            "product":     random.choice(products),
            "amount":      round(random.uniform(50.0, 2000.0), 2),
            "country":     random.choice(countries),
            "landed_at":   datetime.utcnow().isoformat(),
        })

    count = kafka.offsets["orders_raw"]
    logger.success(f"[MODULE 1] {count} events published to orders_raw")
    return count

In [5]:
# =====================================================================
# MODULE 2 — STORAGE (Bronze → Silver → Gold)

In [6]:
# =====================================================================

def module2_storage(kafka: MockKafka) -> dict[str, str]:
    """
    Implements the Medallion Architecture (Delta Lakehouse pattern):

      Bronze — Raw append: every field, no cleaning, with ingestion timestamp.
               In production: Spark Structured Streaming writes directly here
               using .format('delta').mode('append').

      Silver — Cleansed: nulls dropped, invalid rows filtered, types enforced.
               In production: a dbt model with enforced: true transforms Bronze.

      Gold   — Aggregated: business-level metrics ready for BI and AI serving.
               In production: scheduled Spark batch job or dbt mart model.
    """
    logger.info("[MODULE 2] Storage — Bronze → Silver → Gold pipeline...")

    events   = kafka.consume_all("orders_raw")
    payloads = [e["payload"] for e in events]

    for path in [BRONZE_PATH, SILVER_PATH, GOLD_PATH]:
        os.makedirs(path, exist_ok=True)

    # Bronze: raw, no transformation
    bronze_df   = pd.DataFrame(payloads)
    bronze_file = f"{BRONZE_PATH}/orders_{int(datetime.utcnow().timestamp())}.parquet"
    bronze_df.to_parquet(bronze_file, index=False)
    logger.info(f"[MODULE 2] Bronze: {len(bronze_df)} rows → {bronze_file}")

    # Silver: cleansed
    silver_df = bronze_df.dropna()
    silver_df = silver_df[silver_df["amount"] > 0].copy()
    silver_df = silver_df[silver_df["country"].str.match(r"^[A-Z]{2}$", na=False)]
    silver_file = f"{SILVER_PATH}/orders_clean.parquet"
    silver_df.to_parquet(silver_file, index=False)
    dropped = len(bronze_df) - len(silver_df)
    logger.info(f"[MODULE 2] Silver: {len(silver_df)} rows ({dropped} dropped in cleansing)")

    # Gold: revenue by country
    gold_df = (
        silver_df
        .groupby("country")
        .agg(
            total_orders   = ("order_id", "count"),
            total_revenue  = ("amount",   "sum"),
            avg_order_value= ("amount",   "mean"),
        )
        .reset_index()
        .sort_values("total_revenue", ascending=False)
    )
    gold_df["avg_order_value"] = gold_df["avg_order_value"].round(2)
    gold_file = f"{GOLD_PATH}/revenue_by_country.parquet"
    gold_df.to_parquet(gold_file, index=False)
    logger.success("[MODULE 2] Gold layer ready")

    return {"bronze": bronze_file, "silver": silver_file, "gold": gold_file}

In [7]:
# =====================================================================
# MODULE 3 — QUALITY GATE (DAMA + Great Expectations-style)

In [8]:
# =====================================================================

def module3_quality(silver_path: str) -> bool:
    """
    Runs DAMA quality checks on the Silver layer before serving.
    Returns True if all checks pass (pipeline continues), False otherwise.

    In production: replace each check with a Great Expectations expectation:
      expect_column_values_to_not_be_null()
      expect_column_values_to_be_between()
      expect_column_values_to_be_unique()
      expect_column_values_to_match_regex()
    """
    logger.info("[MODULE 3] Quality Gate — validating Silver layer...")
    df     = pd.read_parquet(silver_path)
    passed = True

    checks = [
        (
            "Completeness",
            df.isnull().sum().sum() == 0,
            f"{df.isnull().sum().sum()} null values found",
        ),
        (
            "Accuracy (amount > 0)",
            (df["amount"] > 0).all(),
            "Negative or zero amounts detected",
        ),
        (
            "Uniqueness (order_id)",
            df.duplicated("order_id").sum() == 0,
            f"{df.duplicated('order_id').sum()} duplicate order IDs",
        ),
        (
            "Validity (country ISO-3166)",
            df["country"].str.match(r"^[A-Z]{2}$", na=False).all(),
            "Non-ISO country codes detected",
        ),
        (
            "Timeliness (landed within 10 min)",
            (
                pd.to_datetime(df["landed_at"])
                >= datetime.utcnow() - timedelta(minutes=10)
            ).all(),
            "Records landed outside the 10-minute SLA",
        ),
    ]

    for name, result, fail_detail in checks:
        if result:
            logger.success(f"  [PASSED] {name}")
        else:
            logger.warning(f"  [FAILED] {name}: {fail_detail}")
            passed = False

    return passed

In [9]:
# =====================================================================
# MODULE 4 — SERVING (RAG pipeline over Gold data)

In [10]:
# =====================================================================

def module4_rag_serve(gold_path: str):
    """
    Indexes Gold layer records into ChromaDB and answers analytical queries
    using Retrieval-Augmented Generation.

    In production:
      - Use a persistent Chroma client or Pinecone for the vector index.
      - Call an actual LLM API (OpenRouter, OpenAI, Anthropic) for answers.
      - Implement hybrid search (BM25 + vector) + cross-encoder reranking.
    """
    logger.info("[MODULE 4] Serving — building RAG index over Gold data...")
    df = pd.read_parquet(gold_path)

    # Convert each Gold row into a human-readable document
    documents = [
        (
            f"Country {row.country}: {int(row.total_orders)} orders, "
            f"total revenue ${row.total_revenue:.2f}, "
            f"average order value ${row.avg_order_value:.2f}."
        )
        for _, row in df.iterrows()
    ]

    ef = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
    client = chromadb.Client()
    try:
        client.delete_collection("capstone_gold")
    except Exception:
        pass
    collection = client.create_collection("capstone_gold", embedding_function=ef)
    collection.add(
        ids       = [f"row_{i}" for i in range(len(documents))],
        documents = documents,
    )

    queries = [
        "Which country generated the most revenue?",
        "What is the average order value in Saudi Arabia?",
        "Which markets have the most orders?",
    ]

    embed_model = SentenceTransformer("all-MiniLM-L6-v2")
    for query in queries:
        results  = collection.query(query_texts=[query], n_results=2)
        top_docs = results["documents"][0]
        print(f"\n  🔍 RAG Query: {query}")
        print(f"  📄 Retrieved: {top_docs[0]}")
        print(f"  🤖 [LLM would answer here using the retrieved context]")

    logger.success("[MODULE 4] RAG serving complete")

In [11]:
# =====================================================================
# MODULE 5 — ORCHESTRATION (Airflow-style DAG)

In [12]:
# =====================================================================

def module5_orchestrate():
    """
    Executes all five modules in topological (dependency) order.

    In production: each function below becomes a PythonOperator in
    an Airflow DAG with depends_on_past=False and retries=2.

    DAG shape:
      ingest → storage → quality_gate → serve (if passed)
                                      → quarantine (if failed)

    XCom equivalent: paths dict passes dataset locations between tasks.
    Lineage events are the audit trail — one event per task transition.
    """
    logger.info("[MODULE 5] DAG orchestrator starting...")
    run_id  = str(uuid.uuid4())[:8]
    lineage = []

    def emit(task: str, event: str, details: dict = None):
        lineage.append({
            "runId":  run_id,
            "task":   task,
            "event":  event,
            "ts":     datetime.utcnow().isoformat(),
            **(details or {}),
        })

    kafka = MockKafka()

    # Task 1: Ingest
    emit("ingest", "START")
    count = module1_ingest(kafka)
    emit("ingest", "COMPLETE", {"events_produced": count})

    # Task 2: Storage
    emit("storage", "START")
    paths = module2_storage(kafka)
    emit("storage", "COMPLETE", paths)

    # Task 3: Quality gate (branching)
    emit("quality_gate", "START")
    quality_ok = module3_quality(paths["silver"])

    if quality_ok:
        emit("quality_gate", "COMPLETE")
        # Task 4: Serve (downstream task — only runs if quality passed)
        emit("rag_serve", "START")
        module4_rag_serve(paths["gold"])
        emit("rag_serve", "COMPLETE")
    else:
        emit("quality_gate", "FAIL", {"action": "quarantine — serve step skipped"})
        logger.error("[MODULE 5] Quality gate failed — downstream serve step skipped.")

    # Write lineage audit trail
    os.makedirs("lineage_events", exist_ok=True)
    lineage_path = f"lineage_events/run_{run_id}.json"
    with open(lineage_path, "w") as f:
        json.dump(lineage, f, indent=2)
    logger.info(f"[MODULE 5] Lineage → {lineage_path}")

    return quality_ok

In [13]:
# =====================================================================
# ENTRY POINT

In [14]:
# =====================================================================

def main():
    print("=" * 65)
    print("  Day 5 Capstone — Architecture Integration")
    print("=" * 65)

    success = module5_orchestrate()

    print("\n" + "=" * 65)
    if success:
        print("✅ All 5 modules ran successfully end-to-end.")
        print("")
        print("   Output directories:")
        print(f"   • ./lakehouse/bronze/  — raw ingested records")
        print(f"   • ./lakehouse/silver/  — cleansed, validated records")
        print(f"   • ./lakehouse/gold/    — revenue aggregation by country")
        print(f"   • ./lineage_events/    — pipeline audit trail (JSON)")
    else:
        print("⚠️  Pipeline completed but quality gate failed.")
        print("   Fix the upstream data issues and re-run.")
    print("=" * 65)

    print("\nTeam extension checklist:")
    print("  □ Replace MockKafka → real kafka-python producer/consumer")
    print("  □ Replace pandas parquet writes → PySpark + Delta Lake")
    print("  □ Replace manual checks → Great Expectations checkpoints")
    print("  □ Replace chromadb.Client() → persistent Chroma or Pinecone")
    print("  □ Wrap module5_orchestrate() tasks in an Airflow DAG")
    print("  □ Add OpenLineage HTTP emitter pointing at a Marquez server")
    print("  □ Record a 10-minute demo covering all five modules")

## ▶ Run

In [15]:
main()

  Day 5 Capstone — Architecture Integration


2026-07-31 13:07:57.347 | INFO     | __main__:module5_orchestrate:272 - [MODULE 5] DAG orchestrator starting...
2026-07-31 13:07:57.355 | INFO     | __main__:module1_ingest:57 - [MODULE 1] Ingestion — producing events to Kafka...
2026-07-31 13:07:57.355 | SUCCESS  | __main__:module1_ingest:73 - [MODULE 1] 25 events published to orders_raw
2026-07-31 13:07:57.355 | INFO     | __main__:module2_storage:95 - [MODULE 2] Storage — Bronze → Silver → Gold pipeline...
2026-07-31 13:07:57.373 | INFO     | __main__:module2_storage:107 - [MODULE 2] Bronze: 25 rows → ./lakehouse/bronze/orders_1785481677.parquet
2026-07-31 13:07:57.376 | INFO     | __main__:module2_storage:116 - [MODULE 2] Silver: 25 rows (0 dropped in cleansing)
2026-07-31 13:07:57.382 | SUCCESS  | __main__:module2_storage:133 - [MODULE 2] Gold layer ready
2026-07-31 13:07:57.382 | INFO     | __main__:module3_quality:153 - [MODULE 3] Quality Gate — validating Silver layer...
2026-07-31 13:07:57.408 | SUCCESS  | __main__:module3_qua

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10297.55it/s]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12899.00it/s]



  🔍 RAG Query: Which country generated the most revenue?
  📄 Retrieved: Country EG: 2 orders, total revenue $2338.31, average order value $1169.16.
  🤖 [LLM would answer here using the retrieved context]

  🔍 RAG Query: What is the average order value in Saudi Arabia?
  📄 Retrieved: Country EG: 2 orders, total revenue $2338.31, average order value $1169.16.
  🤖 [LLM would answer here using the retrieved context]

  🔍 RAG Query: Which markets have the most orders?
  📄 Retrieved: Country EG: 2 orders, total revenue $2338.31, average order value $1169.16.
  🤖 [LLM would answer here using the retrieved context]


2026-07-31 13:08:09.466 | SUCCESS  | __main__:module4_rag_serve:251 - [MODULE 4] RAG serving complete
2026-07-31 13:08:09.467 | INFO     | __main__:module5_orchestrate:316 - [MODULE 5] Lineage → lineage_events/run_220340e2.json



✅ All 5 modules ran successfully end-to-end.

   Output directories:
   • ./lakehouse/bronze/  — raw ingested records
   • ./lakehouse/silver/  — cleansed, validated records
   • ./lakehouse/gold/    — revenue aggregation by country
   • ./lineage_events/    — pipeline audit trail (JSON)

Team extension checklist:
  □ Replace MockKafka → real kafka-python producer/consumer
  □ Replace pandas parquet writes → PySpark + Delta Lake
  □ Replace manual checks → Great Expectations checkpoints
  □ Replace chromadb.Client() → persistent Chroma or Pinecone
  □ Wrap module5_orchestrate() tasks in an Airflow DAG
  □ Add OpenLineage HTTP emitter pointing at a Marquez server
  □ Record a 10-minute demo covering all five modules
